# Training Walkthrough

This notebook runs the same training functions used by the CLI, step by step.

1. Configure the experiment.
2. Run SSL pretraining.
3. Run multimodal survival training with the SSL checkpoint.

In [1]:
from pathlib import Path
import os
import sys

import pytorch_lightning as L
import torch
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from src.config import Configuration
from src.data.lightning_datamodule import UPennGBMDataModule
from src.data.ssl_datamodule import SSLDataModule
from src.models import SSLPretrainingLightningModule, MultimodalSurvivalLightningModule
from src.training.device import resolve_lightning_accelerator

CONFIG = Configuration()

SSL_EPOCHS = 20
SURVIVAL_EPOCHS = 20
SSL_CHECKPOINT_NAME = "ssl_checkpoint.pt"
SURVIVAL_CHECKPOINT_NAME = "survival_checkpoint.pt"

print("Configuration ready")

Configuration ready


## Step 1: SSL Pretraining

This uses the Lightning `SSLPretrainingLightningModule` together with the `SSLDataModule`.

In [ ]:
from torch.utils.data import DataLoader, Subset
ssl_dm = SSLDataModule(
    config=CONFIG,
    batch_size=16,
    num_workers=4,
    vol_size=96,
    noise_std=0.05,
    crop_scale=0.85,
)
ssl_dm.setup()

ssl_subset = Subset(ssl_dm.train_ds, range(len(ssl_dm.train_ds)))
ssl_train_loader = DataLoader(
    ssl_subset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
    persistent_workers = True,
    prefetch_factor=4,
)

ssl_module = SSLPretrainingLightningModule(
    embed_dim=512,
    patch_size=16,
    in_channels=4,
    proj_dim=128,
    learning_rate=1e-4,
    weight_decay=1e-4,
    temperature=0.5,
    vit_depth=8,
    num_heads=8,
    dropout=0.1,
    vol_size=96,
)

ssl_accelerator, ssl_devices, ssl_device_label = resolve_lightning_accelerator(CONFIG)
print(f"[SSL] device: {ssl_device_label}")

ssl_checkpoint_cb = ModelCheckpoint(
    dirpath=CONFIG.MODELS_PATH,
    filename="ssl_pretraining_best",
    monitor="train_loss",
    mode="min",
    save_top_k=1,
)

ssl_trainer = L.Trainer(
    max_epochs=SSL_EPOCHS,
    accelerator=ssl_accelerator,
    devices=ssl_devices,
    callbacks=[
        ssl_checkpoint_cb,
        EarlyStopping(
            monitor="train_loss",
            mode="min",
            patience=2,
            min_delta=1e-4,
        ),
    ],
    default_root_dir=CONFIG.LOGS_PATH,
    log_every_n_steps=1,
    enable_progress_bar=True,
     profiler="simple",
)

ssl_trainer.fit(ssl_module, train_dataloaders=ssl_train_loader)

if ssl_checkpoint_cb.best_model_path:
    best_ssl_state = torch.load(ssl_checkpoint_cb.best_model_path, map_location="cpu", weights_only=True)["state_dict"]
    ssl_module.load_state_dict(best_ssl_state)
    print(f"Best SSL checkpoint: {ssl_checkpoint_cb.best_model_path}")

ssl_checkpoint_path = os.path.join(CONFIG.MODELS_PATH, SSL_CHECKPOINT_NAME)
torch.save({"state_dict": ssl_module.state_dict()}, ssl_checkpoint_path)
print(type(ssl_module).__name__)
print(f"SSL checkpoint saved as: {ssl_checkpoint_path}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set `torch.set_fl

[SSL] device: gpu:0 (NVIDIA GeForce RTX 5070)


/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

## Step 2: Survival Training

This uses the Lightning `MultimodalSurvivalLightningModule` together with the existing UPenn GBM data module.

In [ ]:
if "ssl_checkpoint_cb" in globals() and ssl_checkpoint_cb.best_model_path:
    ssl_checkpoint_path = ssl_checkpoint_cb.best_model_path
else:
    ssl_checkpoint_path = os.path.join(CONFIG.MODELS_PATH, SSL_CHECKPOINT_NAME)

print(f"Using SSL checkpoint for survival stage: {ssl_checkpoint_path}")


In [ ]:
survival_dm = UPennGBMDataModule(
    config=CONFIG,
    batch_size=16,
    num_workers=4,
)
survival_dm.setup("fit")
survival_dm.setup("test")

train_subset = Subset(survival_dm.train_ds, range(min(1000, len(survival_dm.train_ds))))
val_subset = Subset(survival_dm.val_ds, range(min(1000, len(survival_dm.val_ds))))
test_subset = Subset(survival_dm.test_ds, range(min(1000, len(survival_dm.test_ds))))

train_loader = DataLoader(
    train_subset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
    persistent_workers = True
)
val_loader = DataLoader(
    val_subset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    persistent_workers = True
)
test_loader = DataLoader(
    test_subset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    persistent_workers = True
)

survival_module = MultimodalSurvivalLightningModule(
    num_classes=len(CONFIG.labels),
    embed_dim=256,
    num_heads=8,
    dropout=0.1,
    in_channels=4,
    patch_size=16,
    vit_depth=4,
    vol_size=96,
    tabular_in=len(survival_dm.train_ds.tabular_cols),
    tabular_tokens=8,
    tabular_hidden=128,
    learning_rate=1e-4,
    weight_decay=1e-4,
    label_smoothing=0.1,
)

survival_module.model.load_pretrained_encoder(ssl_checkpoint_path, strict=False)

survival_accelerator, survival_devices, survival_device_label = resolve_lightning_accelerator(CONFIG)
print(f"[Survival] device: {survival_device_label}")

survival_trainer = L.Trainer(
    max_epochs=SURVIVAL_EPOCHS,
    accelerator=survival_accelerator,
    devices=survival_devices,
    callbacks=[
        ModelCheckpoint(
            dirpath=CONFIG.MODELS_PATH,
            filename="survival_checkpoint_best",
            monitor="val_bacc",
            mode="max",
            save_top_k=1,
        ),
        EarlyStopping(
            monitor="val_bacc",
            mode="max",
            patience=2,
            min_delta=1e-4,
        ),
    ],
    default_root_dir=CONFIG.LOGS_PATH,
    log_every_n_steps=1,
    enable_progress_bar=True,
)

survival_trainer.fit(survival_module, train_dataloaders=train_loader, val_dataloaders=val_loader)
survival_trainer.test(survival_module, dataloaders=test_loader)

survival_checkpoint_path = os.path.join(CONFIG.MODELS_PATH, SURVIVAL_CHECKPOINT_NAME)
torch.save({"state_dict": survival_module.state_dict()}, survival_checkpoint_path)
print(type(survival_module).__name__)
print(f"Survival checkpoint saved as: {survival_checkpoint_path}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type                        | Params | Mode  | FLOPs
--------------------------------------------------------------------------
0 | model     | MultimodalSurvivalPredictor | 10.0 M | train | 0    
1 | criterion | CrossEntropyLoss            | 0      | train | 0    
--------------------------------------------------------------------------
10.0 M    Trainable params
0     

[pretrained encoder] missing=54  unexpected=0
[Survival] device: gpu:0 (NVIDIA GeForce RTX 5070)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc             0.529411792755127
        test_bacc           0.5542483329772949
        test_loss           0.7400573492050171
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
MultimodalSurvivalLightningModule
Survival checkpoint saved as: ../models/survival_checkpoint.pt


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
survival_module = survival_module.to(device)
survival_module.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        try:
            image = batch['image'].to(device)
            tabular = batch['tabular'].to(device)
            labels = batch['label'].to(device)
        except (TypeError, KeyError):
            image, tabular, labels = batch
            image = image.to(device)
            tabular = tabular.to(device)
            labels = labels.to(device)
        preds = survival_module(image, tabular)
        all_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

f1 = f1_score(all_labels, all_preds, average='weighted')
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
accuracy = accuracy_score(all_labels, all_preds)

print(f"Test F1 Score: {f1:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test F1 Score: 0.5126
Test Precision: 0.5389
Test Recall: 0.5294
Test Accuracy: 0.5294


In [ ]:
all_preds

[np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0)]